In [43]:
# Import
import logs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mysql.connector
from mysql.connector import errorcode

import warnings
warnings.filterwarnings("ignore")

In [44]:
class Magasin:

    def __init__(self, host=logs.host, user=logs.user, port=logs.port, password=logs.password,                   database=logs.database,
                 orders=None, customers=None, products=None,
                 sellers=None, order_items=None, geolocation=None):

        self.host = host
        self.user = user
        self.port= port
        self.password = password
        self.database = database

        # Connexion à la base de données
        try:
            self.connection = mysql.connector.connect(
                host=self.host,
                user=self.user,
                port=self.port,
                password=self.password,
                database=self.database
            )
        except mysql.connector.Error as err:
            if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
                print("Erreur: Identifiants incorrects")
            elif err.errno == errorcode.ER_BAD_DB_ERROR:
                print("Erreur: Base de données inexistante")
            else:
                print(err)
            raise

        # Charger les données depuis la base de données ou utiliser les paramètres fournis
        self.orders = orders if orders is not None else pd.read_sql("SELECT * FROM orders", self.connection)
        self.customers = customers if customers is not None else pd.read_sql("SELECT * FROM customers", self.connection)
        self.products = products if products is not None else pd.read_sql("SELECT * FROM products", self.connection)
        self.sellers = sellers if sellers is not None else pd.read_sql("SELECT * FROM sellers", self.connection)
        self.order_items = order_items if order_items is not None else pd.read_sql("SELECT * FROM order_items", self.connection)
        self.geolocation = geolocation if geolocation is not None else pd.read_sql("SELECT * FROM geolocation", self.connection)


    def add_product(self, product_name, product_category, product_platform, product_esrb_rating, product_release_year, product_price, product_weight_g, product_description):

        " Ajoute un produit au magasin "

        cursor = self.connection.cursor()
        query = "INSERT INTO products (product_id,product_name, product_category, product_platform, product_esrb_rating, product_release_year, product_price, product_weight_g, product_description) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)"

        product_id = "PROD_"+"{:05d}".format(self.products.shape[0] + 1)
        cursor.execute(query, (product_id, product_name, product_category, product_platform, product_esrb_rating, product_release_year, product_price, product_weight_g, product_description))

        self.connection.commit()
        cursor.close()

        # Recharger les données
        self.products = pd.read_sql("SELECT * FROM products", self.connection)


    def del_product(self, product_id):

        " Supprime un produit du magasin "

        cursor = self.connection.cursor()
        query = "DELETE FROM products WHERE product_id = %s"
        cursor.execute(query, (product_id,))
        self.connection.commit()
        cursor.close()

        # Recharger les données
        self.products = pd.read_sql("SELECT * FROM products", self.connection)


    def modify_products(self, product_id, new_product_name, new_product_category, new_product_platform, new_product_esrb_rating, new_product_release_year, new_product_price, new_product_weight_g, new_product_description):

        " Modifie les informations d'un produit du magasin "

        cursor = self.connection.cursor()
        query = "UPDATE products SET product_name = %s, product_category = %s, product_platform = %s, product_esrb_rating = %s, product_release_year = %s, product_price = %s, product_weight_g = %s, product_description = %s WHERE product_id = %s"
        cursor.execute(query, (new_product_name, new_product_category, new_product_platform, new_product_esrb_rating, new_product_release_year, new_product_price, new_product_weight_g, new_product_description, product_id))
        self.connection.commit()
        cursor.close()

        # Recharger les données
        self.products = pd.read_sql("SELECT * FROM products", self.connection)

    def filter_products_id(self, product_id):

        " Cherche un produit dans le magasin selon son identifiant "

        return self.products[self.products['product_id'] == product_id]


    def filter_products_name(self, product_name):

        " Cherche un produit dans le magasin selon son nom "

        return self.products[self.products['product_name'] == product_name]


    def filter_products_price(self, min_price, max_price):

        " Cherche les produits dans le magasin selon leur prix "

        return self.products[(self.products['product_price'] >= min_price) & (self.products['product_price'] <= max_price)]


    def filter_products_category(self, product_category):

        " Cherche les produits dans le magasin selon leur catégorie "

        return self.products[self.products['product_category'] == product_category]

    def close_connection(self):

        " Ferme la connection à la base de donnée"

        self.connection.close()


magasin = Magasin()

In [45]:
# Filtre par nom
magasin.filter_products_name("Elden Ring")

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description
0,PROD_00001,Elden Ring,Action RPG,PS5,16+,2022,59.99,790,Game: Elden Ring


In [14]:
# Filtre par prix
magasin.filter_products_price(5, 25)

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description
8,PROD_00009,Dragon Age Inquisition,RPG Aventure,PS4,16+,2014,19.99,267,Game: Dragon Age Inquisition
16,PROD_00017,Uncharted 4,Action Aventure,PS4,16+,2016,19.99,601,Game: Uncharted 4
17,PROD_00018,Tomb Raider 2013,Action Aventure,Multi,18+,2013,19.99,788,Game: Tomb Raider 2013
40,PROD_00041,Persona 4 Golden,JRPG,PC,16+,2008,19.99,265,Game: Persona 4 Golden
48,PROD_00049,Stardew Valley,Simulation,Multi,3+,2016,14.99,214,Game: Stardew Valley
54,PROD_00055,Starbound,Sandbox,Multi,3+,2016,14.99,516,Game: Starbound
56,PROD_00057,Valheim,Survival,PC,10+,2021,19.99,765,Game: Valheim
66,PROD_00067,Rainbow Six Siege,Tactical Shooter,Multi,16+,2015,19.99,772,Game: Rainbow Six Siege
78,PROD_00079,Hollow Knight Silksong,Metroidvania,Switch,12+,2024,14.99,339,Game: Hollow Knight Silksong
80,PROD_00081,Celeste,Platformer,Multi,3+,2018,19.99,596,Game: Celeste


In [11]:
# Filtre par catégorie
magasin.filter_products_category("RPG Aventure")

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description
8,PROD_00009,Dragon Age Inquisition,RPG Aventure,PS4,16+,2014,19.99,267,Game: Dragon Age Inquisition
14,PROD_00015,Final Fantasy VII Remake,RPG Aventure,PS5,16+,2020,49.99,373,Game: Final Fantasy VII Remake


In [37]:
# Ajoute un produit
magasin.add_product("Pokemon Platine", "RPG Aventure", "DS", "3+", "2007", "24.99", "254", "Game : Pokemon Platine")

magasin.filter_products_name("Pokemon Platine")

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description
103,PROD_00104,Pokemon Platine,RPG Aventure,DS,3+,2007,24.99,254,Game : Pokemon Platine


In [39]:
# Supprime un produit
magasin.del_product("PROD_00104")

magasin.filter_products_name("Pokemon Platine")

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description


In [18]:
# Close connection
magasin.close_connection()